# Restaurant search — GPU playground (Approaches 6 & 7)

Evaluate and experiment with the GPU-heavy search approaches against the
209-query benchmark (NDCG@5 / Recall@10 / MRR):

- **Approach 6** — cross-encoder scoring every menu item (`crossencoder`)
- **Approach 7** — local LLM as reader via vLLM (`llm_search.py`)
- plus the cheap baselines (`text`, `enhanced`) and optionally ColBERT / SBERT for comparison

**Before running:** *Runtime → Change runtime type → GPU* (L4 or A100 recommended;
a T4 works with a ≤3B model, see the Approach 7 cell).
The repo is private: add a Colab secret named **`GHTOKEN`** (key icon in the left sidebar,
enable *Notebook access*) holding a GitHub fine-grained token with *Contents: read* on
`ahsank/restaurant-search`. Without a token, the notebook uses an existing checkout at
`/content/restaurant-search` if there is one (e.g. an uploaded and unzipped copy).

Run the install cell **first**, before anything imports `torch`.

In [1]:
# --- Environment detection + code checkout -----------------------------------
import os, subprocess, sys

def detect_environment():
    if 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ:
        return 'Colab'
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'Kaggle'
    return 'Local/Other'

ENV = detect_environment()
print(f'Current environment: {ENV}')

REPO = 'ahsank/restaurant-search'
BRANCH = 'agent/campaign'   # or a feature branch to try unmerged changes

if ENV == 'Colab':
    from google.colab import userdata
    try:
        ghtoken = userdata.get('GHTOKEN')
    except Exception:          # secret not defined / not shared with this notebook
        ghtoken = ''
    root_dir = '/content'
elif ENV == 'Kaggle':
    from kaggle_secrets import UserSecretsClient
    try:
        ghtoken = UserSecretsClient().get_secret('gh-token')
    except Exception:
        ghtoken = ''
    root_dir = '/kaggle/working'
else:
    ghtoken = ''
    root_dir = None            # running from a local checkout: research/colab/ -> repo root

if root_dir is not None and not ghtoken and os.path.isdir(f'{root_dir}/restaurant-search/research'):
    # No token but a checkout is already there (uploaded zip / colab upload): use it as is.
    REPO_DIR = f'{root_dir}/restaurant-search'
    print(f'Using existing checkout at {REPO_DIR}')
elif root_dir is not None:
    REPO_DIR = f'{root_dir}/restaurant-search'
    auth = f'ahsank:{ghtoken}@' if ghtoken else ''
    url = f'https://{auth}github.com/{REPO}.git'
    subprocess.run(['rm', '-rf', REPO_DIR], check=True)
    for branch in dict.fromkeys((BRANCH, 'main')):   # try BRANCH, then main
        r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', branch, url, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'Cloned {REPO}@{branch} -> {REPO_DIR}')
            break
        print(f'Clone of {branch} failed: {r.stderr.replace(ghtoken, "***") if ghtoken else r.stderr}')
    else:
        raise RuntimeError('Could not clone the repo — set the GHTOKEN secret if it is private.')
else:
    REPO_DIR = os.path.abspath('../..')

for p in (f'{REPO_DIR}/research', f'{REPO_DIR}/research/search'):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(REPO_DIR)

Current environment: Colab
Cloned ahsank/restaurant-search@agent/campaign -> /content/restaurant-search


In [3]:
# --- Install GPU dependencies (run before anything imports torch) -------------
# vLLM pins its own torch; sentence-transformers / pylate then reuse that torch.
# Colab prints dependency-conflict warnings for packages we don't use (RAPIDS, ADK) — harmless.
INSTALL_VLLM = True      # Approach 7
INSTALL_PYLATE = False  # ColBERT; conflicts with vLLM, see step 2 below

import subprocess, sys
CORE_PKGS = ['sentence-transformers']
def pip_install(*args):
    """pip install; pip's own output doesn't reach the notebook, so failures are printed here."""
    p = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], capture_output=True, text=True)
    if p.returncode != 0:
        print(f'pip install {" ".join(args)} failed (exit {p.returncode}):\n{p.stderr[-3000:]}')
    return p.returncode == 0

# 1. vLLM first, so pip installs its exact torch / torchvision / transformers. (All three in one call with
#    pylate fails: pylate pins transformers<=5.3, vLLM needs >=5.10, so pip installed nothing at all.)
pip_install(*CORE_PKGS, *(['vllm'] if INSTALL_VLLM else []))
# 2. pylate (ColBERT) only if asked: it pins transformers<=5.3 and sentence-transformers==5.3, so installing
#    it downgrades them and breaks vLLM (and pylate fails to load models with the newer versions). Use
#    INSTALL_PYLATE = True only in a session without vLLM (INSTALL_VLLM = False).
if INSTALL_PYLATE:
    if INSTALL_VLLM:
        print('Skipping pylate: it conflicts with vLLM (set INSTALL_VLLM = False for a ColBERT session).')
        INSTALL_PYLATE = False
    else:
        pip_install('pylate')
if INSTALL_VLLM:
    def ensure_torchvision():
        """Safety net if vLLM's torchvision didn't install: add the release matching vLLM's torch
        (torch 2.N.P <-> torchvision 0.(N+15).P) without touching torch."""
        if subprocess.run([sys.executable, '-c', 'import torchvision'], capture_output=True).returncode == 0:
            return
        probe = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'], capture_output=True, text=True)
        if probe.returncode != 0:
            print('torch not importable:', probe.stderr[-500:]); return
        major, minor, patch = (int(x) for x in probe.stdout.strip().split('+')[0].split('.')[:3])
        pip_install('--no-deps', f'torchvision==0.{minor + 15}.{patch}') or pip_install(f'torchvision==0.{minor + 15}.*')
    ensure_torchvision()
mods = ['torch'] + (['torchvision', 'vllm'] if INSTALL_VLLM else []) + (['pylate'] if INSTALL_PYLATE else [])
check = subprocess.run([sys.executable, '-c', f'import importlib; print(" | ".join(m + " " + importlib.import_module(m).__version__ for m in {mods!r}))'],
                       capture_output=True, text=True)
print(check.stdout.strip() or f'IMPORT CHECK FAILED - experiments needing these will fail:\n{check.stderr[-1500:]}')
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requ

In [2]:
# --- Shared setup: data, benchmark, engine helper ---------------------------
import json, time
from pathlib import Path
import pandas as pd

import engines
from evaluate import evaluate, load_queries
from lib.data import load_data, judged_restaurant_ids

restaurants, menus = load_data(judged_restaurant_ids())
QUERIES = load_queries()
print(f'{len(restaurants)} restaurants, {sum(map(len, menus.values()))} items, {len(QUERIES)} queries')

RESULTS = {}            # engine label -> evaluate() output
SEARCH_FNS = {}         # engine label -> search_fn(query) for the playground cells

def build_engine(engine, model=None, **flags):
    # engines.build_arg_parser pre-reads sys.argv to pick engine-specific flags;
    # point it at our args instead of the kernel's.
    argv = ['--engine', engine] + (['--model', model] if model else [])
    saved, sys.argv = sys.argv, ['notebook', *argv]
    try:
        args = engines.build_arg_parser('notebook').parse_args(argv)
    finally:
        sys.argv = saved
    for k, v in flags.items():
        setattr(args, k, v)
    return engines.build_search_fn(args, restaurants, menus)

def score(label, search_fn):
    t0 = time.time()
    RESULTS[label] = evaluate(search_fn)
    SEARCH_FNS[label] = search_fn
    o = RESULTS[label]['overall']
    print(f"{label}: NDCG@5={o['ndcg@5']:.4f}  Recall@10={o['recall@10']:.4f}  MRR={o['mrr']:.4f}"
          f"  ({time.time() - t0:.0f}s)")

def summary():
    rows = {k: {m: v['overall'][m] for m in ('ndcg@5', 'recall@10', 'mrr')} for k, v in RESULTS.items()}
    return pd.DataFrame(rows).T.sort_values('ndcg@5', ascending=False).round(4)

def by(key):   # key: 'by_category' or 'by_difficulty'
    return pd.DataFrame({k: {c: m['ndcg@5'] for c, m in v[key].items()} for k, v in RESULTS.items()}).round(3)

8 restaurants, 834 items, 209 queries


## Baselines (CPU, seconds)

In [5]:
score('text', build_engine('text'))
score('enhanced', build_engine('enhanced'))

text: NDCG@5=0.4456  Recall@10=0.4708  MRR=0.4692  (0s)
enhanced: NDCG@5=0.5566  Recall@10=0.5589  MRR=0.6050  (7s)


## Approach 6 — cross-encoder (GPU)

Scores every one of the 795 items per query. ~4 s/query on CPU; a few minutes for the
whole benchmark on a GPU. Swap `CE_MODEL` for any `sentence-transformers` CrossEncoder
checkpoint (e.g. `cross-encoder/ms-marco-MiniLM-L12-v2`, `BAAI/bge-reranker-base`).

In [6]:
CE_MODEL = 'cross-encoder/ms-marco-MiniLM-L6-v2'
score(f'crossencoder:{CE_MODEL.split("/")[-1]}', build_engine('crossencoder', CE_MODEL))

Loading cross-encoder 'cross-encoder/ms-marco-MiniLM-L6-v2'... 

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

done
834 items to score per query (exhaustive, no first-stage retriever)
crossencoder:ms-marco-MiniLM-L6-v2: NDCG@5=0.6770  Recall@10=0.5974  MRR=0.8168  (41s)


Optional — ColBERT / SBERT for comparison (set `INSTALL_PYLATE = True` above for ColBERT).

In [7]:
RUN_EXTRA = True
if RUN_EXTRA:
    score('sbert:all-mpnet-base-v2', build_engine('sbert', 'all-mpnet-base-v2'))
    if INSTALL_PYLATE:
        score('colbert', build_engine('colbert'))

Loading model 'all-mpnet-base-v2'... 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

done
Building index... 834 items encoded
sbert:all-mpnet-base-v2: NDCG@5=0.5484  Recall@10=0.4678  MRR=0.7375  (3s)


ImportError: cannot import name 'generate_model_card' from 'sentence_transformers.sentence_transformer.model_card' (/usr/local/lib/python3.13/dist-packages/sentence_transformers/sentence_transformer/model_card.py)

## Approach 7 — local LLM as reader (vLLM, GPU)

One prompt per (query, restaurant): the restaurant's whole menu as a numbered list, the query
last (so vLLM's prefix cache reuses each menu across queries). The model answers with graded
`[item_number, grade]` pairs; restaurants are merged by grade, then rank.

| GPU | Suggested `LLM_MODEL` |
|---|---|
| T4 (16 GB) | `Qwen/Qwen2.5-3B-Instruct`, `microsoft/Phi-3.5-mini-instruct` |
| L4 (24 GB) | `Qwen/Qwen2.5-7B-Instruct` (default) |
| A100 (40/80 GB) | `Qwen/Qwen2.5-14B-Instruct`, `Qwen/Qwen2.5-32B-Instruct-AWQ` |

Gated models (Llama) need an `HF_TOKEN` Colab secret. To try another model, restart the
runtime (vLLM doesn't release GPU memory cleanly) and re-run the setup cells.

In [3]:
import llm_search

LLM_MODEL = 'mistralai/Ministral-8B-Instruct-2410'
# LLM_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
# LLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
GPU_MEM = 0.85    # leave room for the cross-encoder loaded above

if ENV == 'Colab':
    try:
        os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
    except Exception:
        pass

#import torch; torch.cuda.empty_cache()
# llm = llm_search.load_llm(LLM_MODEL, gpu_memory_utilization=GPU_MEM)

In [ ]:
import sys
import io
from google.colab import userdata
from huggingface_hub import HfApi

# --- 1. Verify Hugging Face Token ---
try:
    hf_token = userdata.get('HF_TOKEN')
    api = HfApi(token=hf_token)
    user_info = api.whoami()
    print(f"✅ HF_TOKEN is valid! Logged in as: {user_info.get('username')}")
except Exception as e:
    print(f"⚠️ HF_TOKEN check failed/not set: {e}")
    print("Note: If loading an open model like Qwen2.5, a valid HF_TOKEN might not be strictly required.")

# --- 2. Monkeypatch fileno() for Colab/Jupyter compatibility with vLLM ---
if not hasattr(sys.stdout, 'fileno'):
    # Fallback to standard descriptor 1 if fileno doesn't exist
    sys.stdout.fileno = lambda: 1

try:
    sys.stdout.fileno()
except (io.UnsupportedOperation, AttributeError):
    # Override the ipykernel OutStream block
    sys.stdout.fileno = lambda: 1

if not hasattr(sys.stderr, 'fileno'):
    sys.stderr.fileno = lambda: 2
try:
    sys.stderr.fileno()
except (io.UnsupportedOperation, AttributeError):
    sys.stderr.fileno = lambda: 2

print("✅ Successfully patched standard stream fileno methods. Re-running vLLM initialization...")

# --- 3. Re-run vLLM Load ---
import torch
torch.cuda.empty_cache()
llm = llm_search.load_llm(LLM_MODEL, gpu_memory_utilization=GPU_MEM)

✅ HF_TOKEN is valid! Logged in as: None
✅ Successfully patched standard stream fileno methods. Re-running vLLM initialization...
INFO 10-12 22:05:19 [api_utils.py:273] non-default args: {'max_model_len': 12288, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'mistralai/Ministral-8B-Instruct-2410'}


config.json: 0.00B [00:00, ?B/s]

params.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

INFO 10-12 22:05:22 [config.py:352] Inferred from consolidated*.safetensors files torch.bfloat16 dtype.
INFO 10-12 22:05:42 [model.py:611] Resolved architecture: MistralForCausalLM
INFO 10-12 22:05:42 [model.py:1745] Using max model len 12288
INFO 10-12 22:05:42 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 10-12 22:05:42 [vllm.py:999] Asynchronous scheduling is enabled.
INFO 10-12 22:05:42 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


tekken.json:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

WARNING 10-12 22:05:48 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


In [ ]:
# Full benchmark: 209 queries x 8 restaurants = 1,672 prompts (a few minutes on an L4).
# Rankings JSONL can be replayed anywhere: evaluate.py --engine llm --rankings FILE
tag = LLM_MODEL.split('/')[-1].lower()
RANKINGS = Path(f'research/results/llm/{tag}.jsonl')
llm_search.run(QUERIES, restaurants, menus, LLM_MODEL, RANKINGS, llm=llm)
score(f'llm:{tag}', llm_search.replay_search_fn(RANKINGS))

209 queries x 8 restaurants = 1672 prompts


Rendering conversations:   0%|          | 0/1672 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1672/1672 [1:02:18<00:00,  2.24s/it, est. speed input: 2042.21 toks/s, output: 21.74 toks/s]

Generated in 3747.9s (17.93s per query)
Wrote research/results/llm/ministral-8b-instruct-2410.jsonl
llm:ministral-8b-instruct-2410: NDCG@5=0.4419  Recall@10=0.6998  MRR=0.5326  (0s)


## Compare

In [ ]:
display(summary())
display(by('by_difficulty'))
by('by_category')

,ndcg@5,recall@10,mrr
llm:ministral-8b-instruct-2410,0.4419,0.6998,0.5326


,llm:ministral-8b-instruct-2410
Easy,0.504
Medium,0.363
Hard,0.312


,llm:ministral-8b-instruct-2410
C01,0.284
C02,0.343
C03,0.154
C04,0.374
C05,0.565
C06,0.577
C07,0.233
C08,0.301
C09,0.405
C10,0.085


## Playground — try your own queries

`SEARCH_FNS` holds every engine scored above; the LLM entry replays benchmark rankings,
so the live LLM is wired in separately.

In [ ]:
def show(query, k=8):
    fns = dict(SEARCH_FNS)
    if 'llm' in globals():
        fns[f'llm:{LLM_MODEL.split("/")[-1].lower()} (live)'] = lambda q: llm_search.search(q, llm, restaurants, menus)
    cols = {}
    for label, fn in fns.items():
        if label.startswith('llm:') and '(live)' not in label:
            continue
        hits = fn(query)[:k]
        cols[label] = [f"{h['name']} · {restaurants[h['restaurant_id']]['name']}"
                       + (f" [{h['grade']}]" if 'grade' in h else '') for h in hits]
        cols[label] += [''] * (k - len(cols[label]))
    return pd.DataFrame(cols, index=range(1, k + 1))

show('seafood italian')

INFO 10-07 18:16:56 [hf.py:548] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


,text,enhanced,crossencoder:ms-marco-MiniLM-L6-v2,sbert:all-mpnet-base-v2,colbert,llm:qwen2.5-7b-instruct (live)
1,,Spring Pea Soup · Staple & Fancy,Ceviche De Camarones · Plaza Garcia,Shrimp & Clam Bucatini · Staple & Fancy,Kung Pao Prawns · Great China Restaurant,CajunPasta · Bella Italia [2]
2,,Hamachi Crudo · Staple & Fancy,Seafood Fettuccine · Anthony's HomePort,Bucatini · Staple & Fancy,Seafood Enchiladas · Juanita Cantina,Pappardelle · Staple & Fancy [2]
3,,Prosciutto · Staple & Fancy,Seafood Enchiladas · Plaza Garcia,Pappardelle · Staple & Fancy,Seafood Enchiladas · Plaza Garcia,Shrimp & Clam Bucatini · Staple & Fancy [2]
4,,Fried Oysters · Staple & Fancy,Seafood Enchiladas · Juanita Cantina,Bruschetta · Staple & Fancy,Ceviche De Camarones · Plaza Garcia,Cacio e Pepe Bucatini · Staple & Fancy [2]
5,,Little Gem · Staple & Fancy,Arroz Con Mariscos · Plaza Garcia,Mezze Rigatoni · Staple & Fancy,Seafood Fettuccine · Anthony's HomePort,Spaghetti Carbonara · Bella Italia [1]
6,,Smoked Fish Bruschetta · Staple & Fancy,Camarones al Mojo de Ajo · Plaza Garcia,Calamari · Juanita Cantina,Shrimp with Steamed Tofu · Great China Restaurant,Bucatini · Staple & Fancy [1]
7,,Bucatini · Staple & Fancy,Shrimp & Clam Bucatini · Staple & Fancy,Seafood Fettuccine · Anthony's HomePort,Seafood Omelette · Anthony's HomePort,
8,,Mezze Rigatoni · Staple & Fancy,Camarones Bravos · Plaza Garcia,Cacio e Pepe Bucatini · Staple & Fancy,Seafood Burrito · Juanita Cantina,


In [ ]:
show('something light and vegetarian for lunch')

## Where does the LLM miss? (per-query NDCG@5, worst first)

In [ ]:
llm_key = next((k for k in RESULTS if k.startswith('llm:')), None)
other = next((k for k in RESULTS if k.startswith('crossencoder')), None)
if llm_key:
    pq = pd.DataFrame(RESULTS[llm_key]['per_query']).set_index('query_id')
    if other:
        pq[other] = pd.DataFrame(RESULTS[other]['per_query']).set_index('query_id')['ndcg@5']
    display(pq[pq.n_relevant > 0].sort_values('ndcg@5')
              [['query_text', 'category_name', 'difficulty', 'ndcg@5'] + ([other] if other else [])].head(20))

,query_text,category_name,difficulty,ndcg@5
query_id,,,,
C01-01,Mexican food with shrimp,Cuisine type inference,Medium,0.0
C02-08,fresh cheese salad,Ingredient generalization,Medium,0.0
C02-05,smoked meat sandwich,Ingredient generalization,Hard,0.0
C02-04,stone fruit dessert,Ingredient generalization,Hard,0.0
C04-02,seared fish,Cooking method synonyms,Medium,0.0
C03-04,light fresh starter,Attribute / implied meaning,Medium,0.0
C03-02,something smoky,Attribute / implied meaning,Hard,0.0
C02-10,citrus fish,Ingredient generalization,Medium,0.0
C03-07,umami dish,Attribute / implied meaning,Hard,0.0


In [ ]:
# Inspect the raw LLM answer for one query (per restaurant)
QID = 'C03-01'
row = next(json.loads(l) for l in open(RANKINGS) if json.loads(l).get('query_id') == QID)
print(row['query_text'])
for rid, text in row['raw'].items():
    print(f'  {rid:<24} {text[:200]}')

spicy shrimp
  anthonys-homeport        [[2, 2], [39, 1]]
  bella-italia             []
  due-cucina               []
  great-china-restaurant   [[115, 1], [116, 1], [117, 1]]
  juanita-cantina          [[90, 2], [39, 1], [91, 1]]
  mendocino-farms          []
  plaza-garcia             [[87, 1], [107, 2], [118, 2]]
  staple-and-fancy         [[2, 1], [23, 1]]


## Save results

Download `research/results/` (or copy it to Drive) to keep rankings and metrics:
commit the `.jsonl` files so others can replay them with `evaluate.py --engine llm --rankings`.

In [ ]:
summary().to_csv('research/results/summary.csv')
if ENV == 'Colab':
    !cd research && zip -qr /content/results.zip results
    from google.colab import files
    files.download('/content/results.zip')

In [5]:
# --- Score one category, or all of them, for one engine; terminal-style output ---
from collections import defaultdict
from evaluate import evaluate, load_queries, load_relevance, print_report

JUDGMENTS = load_relevance()   # {(restaurant_id, query_id): {item_name: relevance}}

AGENT_DIR = Path('research/agent')

def find_rankings(name=None):
    """Rankings JSONL for `name`: a path, an agent experiment name (its --rankings, e.g.
    'llm-gemma-3-12b-it' or a hybrid's name), or a file name in research/agent/runs/ or
    research/results/llm/ (e.g. 'gemma-3-12b-it'). None = this notebook's own full run (RANKINGS)."""
    if name is None:
        if 'RANKINGS' not in globals():
            raise RuntimeError('no full run in this notebook yet: pass model=<experiment or rankings file>')
        return RANKINGS
    result = AGENT_DIR / 'results' / f'{name}.json'
    if result.exists():
        args = json.loads(result.read_text())['eval_args']
        if '--rankings' in args:
            return Path(args[args.index('--rankings') + 1])
    stem = name[:-len('.jsonl')] if name.endswith('.jsonl') else name
    for path in (Path(name), AGENT_DIR / 'runs' / f'{stem}.jsonl', Path('research/results/llm') / f'{stem}.jsonl'):
        if path.is_file():
            return path
    files = sorted(f.stem for f in (AGENT_DIR / 'runs').glob('*.jsonl'))
    raise FileNotFoundError(f'no rankings for {name!r}; files in research/agent/runs/: {", ".join(files)}')

def experiment_fn(name):
    """search_fn for research/agent/results/<name>.json, built with the same --engine, --model,
    --rankings and flags as the agent's run (cached in SEARCH_FNS under the experiment's name)."""
    if name not in SEARCH_FNS:
        argv = json.loads((AGENT_DIR / 'results' / f'{name}.json').read_text())['eval_args']
        saved, sys.argv = sys.argv, ['notebook', *argv]
        try:
            args, _ = engines.build_arg_parser('notebook').parse_known_args(argv)
        finally:
            sys.argv = saved
        SEARCH_FNS[name] = engines.build_search_fn(args, restaurants, menus)
    return SEARCH_FNS[name]

def _search_fn(engine, model, queries, live):
    if engine == 'llm':
        if not live:
            return llm_search.replay_search_fn(find_rankings(model))
        if 'llm' not in globals():
            raise RuntimeError('load the model first (the load_llm cell)')
        if model and model != LLM_MODEL:
            raise ValueError(f'the loaded model is {LLM_MODEL}; use live=False to replay {model!r}, '
                             'or set LLM_MODEL and rerun the load_llm cell')
        rows = llm_search.rank_queries(llm, queries, restaurants, menus)   # one batch
        by_text = {r['query_text']: r['results'] for r in rows}
        return lambda q: by_text[q]
    key = f'{engine}:{model}' if model else engine
    if key not in SEARCH_FNS:
        SEARCH_FNS[key] = build_engine(engine, model)
    return SEARCH_FNS[key]

def _avg(rows, key):
    rows = [r for r in rows if r['n_relevant'] > 0]
    return sum(r[key] for r in rows) / len(rows) if rows else 0.0

def _print_query_table(rows):
    w = max(len(r['query_text']) for r in rows)
    print(f"  {'query':<7} {'D':<1} {'NDCG@5':>6} {'R@10':>6} {'MRR':>6} {'#rel':>4}  text")
    print(f"  {'-'*7} {'-'} {'-'*6} {'-'*6} {'-'*6} {'-'*4}  {'-'*w}")
    for r in rows:
        print(f"  {r['query_id']:<7} {r['difficulty'][0]} {r['ndcg@5']:6.3f} {r['recall@10']:6.3f} "
              f"{r['mrr']:6.3f} {r['n_relevant']:4d}  {r['query_text']}")
    print()

def _print_details(rows, search_fn, k):
    # top-k per query: [2]/[1] = judged relevant, [ ] = not; then judged items it missed
    for r in rows:
        if r['n_relevant'] == 0:
            continue
        qid = r['query_id']
        rels = {(rid, name): rel for (rid, q), items in JUDGMENTS.items() if q == qid
                for name, rel in items.items()}
        hits = search_fn(r['query_text'])
        print(f"  {qid}  NDCG@5={r['ndcg@5']:.3f}  «{r['query_text']}»")
        for i, h in enumerate(hits[:k], 1):
            rel = rels.get((h['restaurant_id'], h['name']), 0)
            print(f"     {i}. [{rel or ' '}] {h['name']} · {restaurants[h['restaurant_id']]['name']}")
        shown = {(h['restaurant_id'], h['name']) for h in hits[:k]}
        missed = sorted(((rel, n, rid) for (rid, n), rel in rels.items() if (rid, n) not in shown),
                        reverse=True)
        for rel, n, rid in missed[:5]:
            print(f"        missed [{rel}] {n} · {restaurants[rid]['name']}")
        print()

def score_category(engine=None, category=None, difficulty=None, model=None,
                   live=True, details=True, k=5, report=None, return_results=False, experiment=None):
    """
    engine: 'text', 'enhanced', 'sbert', 'colbert', 'crossencoder', or 'llm'
    category: e.g. 'C01'; None or 'all' = every category, printed one section per category
    difficulty: 'Easy' / 'Medium' / 'Hard' (None = all)
    model: model name for sbert / colbert / crossencoder; for 'llm' with live=False, the rankings to
           replay: an experiment name ('llm-gemma-3-12b-it'), a file in research/agent/runs/
           ('gemma-3-12b-it') or a path (None = this notebook's full run, RANKINGS)
    live: for 'llm', True = run the loaded model now; False = replay a rankings file (see model)
    experiment: an agent experiment name (research/agent/results/<name>.json), e.g. a hybrid;
                rebuilds it with the agent's exact flags and ignores engine / model / live
    details: print each query's top-k results and missed relevant items
    report: print evaluate.py's by-category / by-difficulty breakdown at the end
            (default: on for all categories, off for a single one)
    """
    if category == 'all':
        category = None
    if report is None:
        report = category is None

    queries = load_queries(category=category, difficulty=difficulty)
    if not queries:
        raise ValueError(f'no queries for category={category!r} difficulty={difficulty!r}')

    if experiment:
        search_fn, label = experiment_fn(experiment), experiment
    elif engine is None:
        raise ValueError('give an engine or experiment=<name>')
    else:
        search_fn = _search_fn(engine, model, queries, live)
        label = engine + (f':{model}' if model else '')
    res = evaluate(search_fn, category=category, difficulty=difficulty)

    o = res['overall']
    print('=' * 78)
    print(f"{label}  category={category or 'all'}  difficulty={difficulty or 'all'}  "
          f"({o['n_with_judgments']}/{o['n_queries']} queries judged)")
    print(f"  NDCG@5={o['ndcg@5']:.4f}  Recall@10={o['recall@10']:.4f}  MRR={o['mrr']:.4f}")
    print('=' * 78 + '\n')

    by_cat = defaultdict(list)
    for r in res['per_query']:
        by_cat[r['category_id']].append(r)

    for cat in sorted(by_cat):
        rows = sorted(by_cat[cat], key=lambda r: r['ndcg@5'])   # worst first
        if len(by_cat) > 1:
            print('-' * 78)
            print(f"{cat}  {rows[0]['category_name']}  (n={len(rows)})  "
                  f"NDCG@5={_avg(rows, 'ndcg@5'):.3f}  Recall@10={_avg(rows, 'recall@10'):.3f}  "
                  f"MRR={_avg(rows, 'mrr'):.3f}")
            print('-' * 78)
        _print_query_table(rows)
        if details:
            _print_details(rows, search_fn, k)

    if report:
        print_report(res)
    return res if return_results else None
                      # every


In [ ]:

# Examples:
score_category('llm', 'all', live=False)              # every category, replaying this notebook's full-run RANKINGS
# score_category('llm', 'all', model='llm-gemma-3-12b-it', live=False)   # replay an agent experiment's rankings
# score_category('llm', 'all')                        # run the loaded model now

In [ ]:
score_category('colbert', 'all')

In [6]:
# --- Score any agent experiment (LLM, hybrid, ...) from its saved rankings ---------------
# Experiment names are the files in research/agent/results/ (see research/agent/LEADERBOARD.md).
score_category(experiment='hybrid-llama4-scout-bge-reranker-v2-m3')
# score_category(category='C10', experiment='hybrid-qwen2.5-32b-bge-reranker-v2-m3')
# score_category('llm', 'all', model='llm-gemma-3-12b-it', live=False)  # replay one LLM's rankings
# score_category('llm', 'C03', model='gpt-oss-20b', live=False)          # a file in research/agent/runs/

Loading LLM rankings from 'research/agent/runs/qwen2.5-32b-instruct.jsonl'... 209 queries loaded
Loading CrossEncoder model 'BAAI/bge-reranker-v2-m3'... 

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

done
hybrid:BAAI/bge-reranker-v2-m3  category=all  difficulty=all  (206/209 queries judged)
  NDCG@5=0.8204  Recall@10=0.7937  MRR=0.8892

------------------------------------------------------------------------------
C01  Cuisine type inference  (n=8)  NDCG@5=0.649  Recall@10=0.667  MRR=0.906
------------------------------------------------------------------------------
  query   D NDCG@5   R@10    MRR #rel  text
  ------- - ------ ------ ------ ----  -----------------------------
  C01-08  H  0.170  0.364  1.000   17  fine dining pasta
  C01-04  H  0.204  0.500  0.250   10  Tex-Mex combination plate
  C01-01  M  0.509  0.600  1.000   23  Mexican food with shrimp
  C01-03  M  0.723  0.368  1.000   22  Latin seafood dish
  C01-02  M  0.861  1.000  1.000   15  Italian pasta with meat sauce
  C01-06  H  0.893  1.000  1.000   12  taqueria style tacos
  C01-07  H  0.915  0.500  1.000   25  rustic Italian comfort food
  C01-05  M  0.920  1.000  1.000   14  farm to table sandwich

  C01-08  